
# Major Index Tuning Synthesis And Top-5 Selection

This notebook consolidates the completed hyperparameter-tuning results from notebooks `03` and `04`.

The active selection rule is setup-specific and horizon-specific:

- each setup is one `model x window_size` pair
- each setup is ranked by the average `abs_error_pct` over the four indices during the `2022` tuning span
- the primary output is the top 5 configurations for each setup
- the final downstream reruns use only these top-5 configurations

For the recurrent deep-learning models, the notebook also reports an effective-config view. This matters because the current recurrent implementation does not consume the `activation` setting, so tied activation variants are collapsed for selection and reporting.


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def resolve_project_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Run this notebook from the project root or its notebooks directory.')


PROJECT_ROOT = resolve_project_root()
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / '10_major_index_tuning_selection_summary'
TABLE_DIR = OUTPUT_ROOT / 'tables'
FIG_DIR = OUTPUT_ROOT / 'figures'
for directory in [OUTPUT_ROOT, TABLE_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

ML_SCORES_PATH = PROJECT_ROOT / 'outputs' / '03_major_index_classical_ml_rolling' / 'tables' / 'scores_summary.csv'
DL_SCORES_PATH = PROJECT_ROOT / 'outputs' / '04_major_index_deep_learning_rolling' / 'tables' / 'scores_summary.csv'
ML_GRID_PATH = PROJECT_ROOT / 'outputs' / '03_major_index_classical_ml_rolling' / 'tables' / 'grid_profile.csv'
DL_GRID_PATH = PROJECT_ROOT / 'outputs' / '04_major_index_deep_learning_rolling' / 'tables' / 'grid_profile.csv'

PRIMARY_METRIC = 'tune_abs_error_pct_mean'
SECONDARY_METRICS = ['tune_close_RMSE_pct', 'tune_close_RMSE', 'config_index']
TOP_K = 5

ML_RANKING_PATH = TABLE_DIR / 'ml_setup_ranking.csv'
DL_RANKING_PATH = TABLE_DIR / 'dl_setup_ranking.csv'
ML_TOP5_PATH = TABLE_DIR / 'ml_top5_manifest.csv'
DL_TOP5_PATH = TABLE_DIR / 'dl_top5_manifest.csv'
ML_SELECTED_PATH = TABLE_DIR / 'ml_selected_best.csv'
DL_SELECTED_PATH = TABLE_DIR / 'dl_selected_best.csv'
COMBINED_TOP5_PATH = TABLE_DIR / 'combined_top5_manifest.csv'
COMBINED_SELECTED_PATH = TABLE_DIR / 'combined_selected_best.csv'
SELECTION_BUNDLE_PATH = OUTPUT_ROOT / 'selection_bundle.joblib'

In [2]:
def parse_config_json(value: str) -> dict:
    payload = json.loads(value)
    return payload


def format_effective_summary(model_name: str, family: str, config: dict) -> str:
    if family == 'dl' and model_name in {'RNN', 'LSTM', 'GRU'}:
        parts = [
            f"hid={int(config['hidden_dim'])}",
            f"layers={int(config['num_layers'])}",
            f"drop={float(config['dropout']):.2f}",
            f"lr={float(config['learning_rate']):.0e}",
            f"bs={int(config['batch_size'])}",
            f"ep={int(config['epoch'])}",
            f"wd={float(config['weight_decay']):.0e}",
        ]
        return ' '.join(parts)
    if family == 'dl':
        parts = [
            f"hid={int(config['hidden_dim'])}",
            f"layers={int(config['num_layers'])}",
            f"drop={float(config['dropout']):.2f}",
            f"lr={float(config['learning_rate']):.0e}",
            f"bs={int(config['batch_size'])}",
            f"ep={int(config['epoch'])}",
            f"act={config['activation']}",
            f"wd={float(config['weight_decay']):.0e}",
        ]
        return ' '.join(parts)
    return ' '.join(f'{k}={v}' for k, v in config.items())



def effective_config_payload(model_name: str, family: str, config: dict) -> dict:
    payload = dict(config)
    if family == 'dl' and model_name in {'RNN', 'LSTM', 'GRU'}:
        payload.pop('activation', None)
    return payload



def build_setup_ranking(scores: pd.DataFrame, family: str) -> pd.DataFrame:
    frame = scores.copy()
    frame['family'] = family
    frame['model_name'] = frame['model']
    frame['setup_key'] = frame['model_name'] + '_w' + frame['window_size'].astype(str)
    configs = frame['config_json'].map(parse_config_json)
    frame['effective_config_json'] = [json.dumps(effective_config_payload(m, family, cfg), sort_keys=True) for m, cfg in zip(frame['model_name'], configs)]
    frame['effective_config_summary'] = [format_effective_summary(m, family, cfg) for m, cfg in zip(frame['model_name'], configs)]

    group_cols = ['family', 'model_name', 'window_size', 'setup_key', 'effective_config_json', 'effective_config_summary']
    grouped = (
        frame
        .sort_values(['model_name', 'window_size', 'config_index'])
        .groupby(group_cols, as_index=False)
        .agg(
            avg_tune_abs_error_pct_mean=(PRIMARY_METRIC, 'mean'),
            avg_tune_close_RMSE_pct=('tune_close_RMSE_pct', 'mean'),
            avg_tune_close_RMSE=('tune_close_RMSE', 'mean'),
            avg_tune_close_MAE=('tune_close_MAE', 'mean'),
            avg_tune_close_R2=('tune_close_R2', 'mean'),
            n_indices=('index_key', 'nunique'),
            config_index=('config_index', 'min'),
            config_key=('config_key', 'first'),
            config_summary=('config_summary', 'first'),
            config_json=('config_json', 'first'),
            source_rows=('case_key', 'size'),
        )
    )
    grouped = grouped.sort_values(
        ['model_name', 'window_size', 'avg_tune_abs_error_pct_mean', 'avg_tune_close_RMSE_pct', 'avg_tune_close_RMSE', 'config_index']
    ).reset_index(drop=True)
    grouped['selection_rank'] = grouped.groupby(['model_name', 'window_size']).cumcount() + 1
    grouped['is_top5'] = grouped['selection_rank'] <= TOP_K
    grouped['is_selected_best'] = grouped['selection_rank'] == 1
    return grouped



def build_index_detail(scores: pd.DataFrame, ranking: pd.DataFrame, family: str) -> pd.DataFrame:
    frame = scores.copy()
    frame['family'] = family
    frame['model_name'] = frame['model']
    configs = frame['config_json'].map(parse_config_json)
    frame['effective_config_json'] = [json.dumps(effective_config_payload(m, family, cfg), sort_keys=True) for m, cfg in zip(frame['model_name'], configs)]
    detail = frame.merge(
        ranking[
            [
                'family', 'model_name', 'window_size', 'effective_config_json', 'selection_rank', 'is_top5', 'is_selected_best'
            ]
        ],
        on=['family', 'model_name', 'window_size', 'effective_config_json'],
        how='left',
    )
    return detail.sort_values(['family', 'model_name', 'window_size', 'selection_rank', 'index_key']).reset_index(drop=True)



def plot_tuning_landscape(ranking: pd.DataFrame, family_label: str, out_path: Path) -> None:
    models = list(ranking['model_name'].drop_duplicates())
    windows = sorted(ranking['window_size'].drop_duplicates())
    fig, axes = plt.subplots(len(models), len(windows), figsize=(4.9 * len(windows), 3.6 * len(models)), squeeze=False)
    for i, model_name in enumerate(models):
        for j, window_size in enumerate(windows):
            ax = axes[i][j]
            subset = ranking[(ranking['model_name'] == model_name) & (ranking['window_size'] == window_size)].copy()
            ax.scatter(
                subset['avg_tune_abs_error_pct_mean'],
                subset['avg_tune_close_RMSE_pct'],
                s=np.where(subset['is_top5'], 56, 22),
                c=np.where(subset['is_top5'], '#0E7C7B', '#A7B3C4'),
                alpha=0.9,
                edgecolor='white',
                linewidth=0.4,
            )
            top5 = subset[subset['is_top5']].copy()
            for _, row in top5.iterrows():
                ax.text(
                    row['avg_tune_abs_error_pct_mean'],
                    row['avg_tune_close_RMSE_pct'],
                    str(int(row['selection_rank'])),
                    fontsize=8,
                    ha='center',
                    va='center',
                    color='white',
                    fontweight='bold',
                )
            ax.set_title(f'{model_name} | w={window_size}')
            ax.set_xlabel('Avg tuning abs % error across indices')
            ax.set_ylabel('Avg tuning RMSE % across indices')
            ax.grid(alpha=0.25)
    fig.suptitle(f'{family_label}: tuning landscape and selected top-5 configs', fontsize=14, y=1.02)
    fig.tight_layout()
    fig.savefig(out_path, dpi=220, bbox_inches='tight')
    plt.close(fig)



def plot_top5_bars(top5: pd.DataFrame, family_label: str, out_path: Path) -> None:
    setups = top5['setup_key'].drop_duplicates().tolist()
    fig, axes = plt.subplots(len(setups), 1, figsize=(10.5, 2.8 * len(setups)), squeeze=False)
    for idx, setup_key in enumerate(setups):
        ax = axes[idx][0]
        subset = top5[top5['setup_key'] == setup_key].sort_values('selection_rank')
        ax.barh(subset['selection_rank'].astype(str), subset['avg_tune_abs_error_pct_mean'], color='#2F5496')
        ax.invert_yaxis()
        ax.set_title(f'{setup_key}: top-5 tuning configs')
        ax.set_xlabel('Avg tuning abs % error across indices')
        ax.set_ylabel('Rank')
        for y, value in zip(subset['selection_rank'].astype(str), subset['avg_tune_abs_error_pct_mean']):
            ax.text(value, y, f' {value:.3f}', va='center', fontsize=8, color='#233049')
    fig.suptitle(f'{family_label}: top-5 setup-specific selections', fontsize=14, y=1.01)
    fig.tight_layout()
    fig.savefig(out_path, dpi=220, bbox_inches='tight')
    plt.close(fig)

In [3]:
ml_scores = pd.read_csv(ML_SCORES_PATH)
dl_scores = pd.read_csv(DL_SCORES_PATH)
ml_grid = pd.read_csv(ML_GRID_PATH)
dl_grid = pd.read_csv(DL_GRID_PATH)

ml_ranking = build_setup_ranking(ml_scores, family='ml')
dl_ranking = build_setup_ranking(dl_scores, family='dl')
ml_detail = build_index_detail(ml_scores, ml_ranking, family='ml')
dl_detail = build_index_detail(dl_scores, dl_ranking, family='dl')

ml_top5 = ml_ranking[ml_ranking['is_top5']].copy().reset_index(drop=True)
dl_top5 = dl_ranking[dl_ranking['is_top5']].copy().reset_index(drop=True)
ml_selected = ml_ranking[ml_ranking['is_selected_best']].copy().reset_index(drop=True)
dl_selected = dl_ranking[dl_ranking['is_selected_best']].copy().reset_index(drop=True)
combined_top5 = pd.concat([ml_top5, dl_top5], ignore_index=True)
combined_selected = pd.concat([ml_selected, dl_selected], ignore_index=True)

ml_ranking.to_csv(ML_RANKING_PATH, index=False)
dl_ranking.to_csv(DL_RANKING_PATH, index=False)
ml_top5.to_csv(ML_TOP5_PATH, index=False)
dl_top5.to_csv(DL_TOP5_PATH, index=False)
ml_selected.to_csv(ML_SELECTED_PATH, index=False)
dl_selected.to_csv(DL_SELECTED_PATH, index=False)
combined_top5.to_csv(COMBINED_TOP5_PATH, index=False)
combined_selected.to_csv(COMBINED_SELECTED_PATH, index=False)
ml_detail.to_csv(TABLE_DIR / 'ml_index_detail.csv', index=False)
dl_detail.to_csv(TABLE_DIR / 'dl_index_detail.csv', index=False)
ml_grid.to_csv(TABLE_DIR / 'ml_grid_profile.csv', index=False)
dl_grid.to_csv(TABLE_DIR / 'dl_grid_profile.csv', index=False)

joblib.dump(
    {
        'ml_ranking': ml_ranking,
        'dl_ranking': dl_ranking,
        'ml_top5': ml_top5,
        'dl_top5': dl_top5,
        'ml_selected': ml_selected,
        'dl_selected': dl_selected,
        'combined_top5': combined_top5,
        'combined_selected': combined_selected,
        'primary_metric': PRIMARY_METRIC,
        'top_k': TOP_K,
    },
    SELECTION_BUNDLE_PATH,
)

print('Selection bundle written to:', SELECTION_BUNDLE_PATH)
print('')
print('ML selected-best rows:', len(ml_selected))
print('DL selected-best rows:', len(dl_selected))
print('')
display(ml_grid)
display(dl_grid)

Selection bundle written to: /Users/jimyhc/Desktop/research/quantum_index/root/outputs/10_major_index_tuning_selection_summary/selection_bundle.joblib

ML selected-best rows: 8
DL selected-best rows: 8



,model_name,n_configs,grid_spec
0,GradientBoosting,36,"n_estimators=[100, 300, 500] | learning_rate=[..."
1,RandomForest,36,"n_estimators=[100, 300, 500] | max_depth=[None..."
2,SVR,36,"C=[0.1, 0.5, 2.5, 10.0] | epsilon=[0.01, 0.03,..."
3,XGBoost,36,"n_estimators=[100, 300, 500] | learning_rate=[..."


,model_name,n_configs,grid_spec
0,GRU,243,"hidden_dim=[32, 64, 128] | num_layers=[1, 2, 3..."
1,LSTM,243,"hidden_dim=[32, 64, 128] | num_layers=[1, 2, 3..."
2,MLP,243,"hidden_dim=[32, 64, 128] | num_layers=[1, 2, 3..."
3,RNN,243,"hidden_dim=[32, 64, 128] | num_layers=[1, 2, 3..."


In [4]:
plot_tuning_landscape(ml_ranking, 'Classical ML', FIG_DIR / '01_ml_tuning_landscape.png')
plot_tuning_landscape(dl_ranking, 'Deep Learning', FIG_DIR / '02_dl_tuning_landscape.png')
plot_top5_bars(ml_top5, 'Classical ML', FIG_DIR / '03_ml_top5_bars.png')
plot_top5_bars(dl_top5, 'Deep Learning', FIG_DIR / '04_dl_top5_bars.png')

print('Saved figures:')
for path in sorted(FIG_DIR.glob('*.png')):
    print('-', path.name)

Saved figures:
- 01_ml_tuning_landscape.png
- 02_dl_tuning_landscape.png
- 03_ml_top5_bars.png
- 04_dl_top5_bars.png



## Selected Configurations

The tables below are the active selection manifests for the rest of the `10s` workflow:

- `combined_top5_manifest.csv`: top 5 configs for each `model x horizon` setup
- `combined_selected_best.csv`: the single top-ranked config for each `model x horizon` setup

These manifests are the source of truth for the later full-span rerun notebooks.


In [5]:
summary_cols = [
    'family', 'model_name', 'window_size', 'selection_rank', 'config_key', 'effective_config_summary',
    'avg_tune_abs_error_pct_mean', 'avg_tune_close_RMSE_pct', 'n_indices'
]

display(ml_top5[summary_cols].head(20))
display(dl_top5[summary_cols].head(20))
display(ml_selected[summary_cols].sort_values(['model_name', 'window_size']))
display(dl_selected[summary_cols].sort_values(['model_name', 'window_size']))

,family,model_name,window_size,selection_rank,config_key,effective_config_summary,avg_tune_abs_error_pct_mean,avg_tune_close_RMSE_pct,n_indices
0,ml,GradientBoosting,10,1,4279e6e80447c71a,learning_rate=0.04 max_depth=2 n_estimators=10...,1.644619,2.004291,4
1,ml,GradientBoosting,10,2,e8499da06c5ad0f3,learning_rate=0.16 max_depth=2 n_estimators=10...,1.645851,2.014674,4
2,ml,GradientBoosting,10,3,3eb504562167ef28,learning_rate=0.16 max_depth=2 n_estimators=50...,1.645868,2.014695,4
3,ml,GradientBoosting,10,4,1bfbc6c29848f690,learning_rate=0.16 max_depth=2 n_estimators=30...,1.645868,2.014695,4
4,ml,GradientBoosting,10,5,5708720c56ae8c36,learning_rate=0.04 max_depth=3 n_estimators=10...,1.647700,2.009166,4
5,ml,GradientBoosting,20,1,dcb8076b850ef5d4,learning_rate=0.08 max_depth=2 n_estimators=10...,1.681913,2.089605,4
6,ml,GradientBoosting,20,2,443ba1a5a28c4023,learning_rate=0.08 max_depth=2 n_estimators=30...,1.686191,2.098515,4
7,ml,GradientBoosting,20,3,8ad8b1cebd809d2d,learning_rate=0.08 max_depth=2 n_estimators=50...,1.686300,2.098671,4
8,ml,GradientBoosting,20,4,0c65dee93bf18bc7,learning_rate=0.04 max_depth=3 n_estimators=30...,1.694941,2.098843,4
9,ml,GradientBoosting,20,5,df972d756040e157,learning_rate=0.04 max_depth=3 n_estimators=50...,1.695087,2.099045,4


,family,model_name,window_size,selection_rank,config_key,effective_config_summary,avg_tune_abs_error_pct_mean,avg_tune_close_RMSE_pct,n_indices
0,dl,GRU,10,1,98ffd95c2936a288,hid=128 layers=3 drop=0.00 lr=3e-04 bs=16 ep=3...,1.545700,1.882489,4
1,dl,GRU,10,2,cafef798b3a445c6,hid=128 layers=3 drop=0.00 lr=3e-04 bs=16 ep=3...,1.545701,1.882492,4
2,dl,GRU,10,3,f63675e18c4c2102,hid=128 layers=3 drop=0.00 lr=3e-04 bs=16 ep=3...,1.545702,1.882493,4
3,dl,GRU,10,4,4e243d8835e0f454,hid=128 layers=2 drop=0.00 lr=3e-04 bs=16 ep=3...,1.558440,1.901438,4
4,dl,GRU,10,5,675f474344b91b59,hid=128 layers=2 drop=0.00 lr=3e-04 bs=16 ep=3...,1.558441,1.901441,4
5,dl,GRU,20,1,4e243d8835e0f454,hid=128 layers=2 drop=0.00 lr=3e-04 bs=16 ep=3...,1.527046,1.903072,4
6,dl,GRU,20,2,675f474344b91b59,hid=128 layers=2 drop=0.00 lr=3e-04 bs=16 ep=3...,1.527046,1.903073,4
7,dl,GRU,20,3,4175d8c3b62e9175,hid=128 layers=2 drop=0.00 lr=3e-04 bs=16 ep=3...,1.527046,1.903073,4
8,dl,GRU,20,4,1c64976ca6bd8834,hid=128 layers=1 drop=0.00 lr=3e-04 bs=16 ep=3...,1.536276,1.905397,4
9,dl,GRU,20,5,e8d6dc94d91b807d,hid=128 layers=1 drop=0.00 lr=3e-04 bs=16 ep=3...,1.536277,1.905400,4


,family,model_name,window_size,selection_rank,config_key,effective_config_summary,avg_tune_abs_error_pct_mean,avg_tune_close_RMSE_pct,n_indices
0,ml,GradientBoosting,10,1,4279e6e80447c71a,learning_rate=0.04 max_depth=2 n_estimators=10...,1.644619,2.004291,4
1,ml,GradientBoosting,20,1,dcb8076b850ef5d4,learning_rate=0.08 max_depth=2 n_estimators=10...,1.681913,2.089605,4
2,ml,RandomForest,10,1,cb02c1136792f10c,max_depth=None max_features=0.8 min_samples_le...,1.638683,1.979725,4
3,ml,RandomForest,20,1,4fed66cd3f5ddd7a,max_depth=4 max_features=0.8 min_samples_leaf=...,1.726920,2.137143,4
4,ml,SVR,10,1,cc4c82a5255cfd43,C=10.0 epsilon=0.1 gamma=0.01,1.612836,1.979841,4
5,ml,SVR,20,1,00f2c7c3b40f1730,C=10.0 epsilon=0.01 gamma=0.01,1.635696,2.032461,4
6,ml,XGBoost,10,1,6c6dc62e47ce264e,learning_rate=0.08 max_depth=3 n_estimators=30...,1.609594,1.993257,4
7,ml,XGBoost,20,1,6ed5493d5a698f10,learning_rate=0.08 max_depth=3 n_estimators=50...,1.712847,2.124949,4


,family,model_name,window_size,selection_rank,config_key,effective_config_summary,avg_tune_abs_error_pct_mean,avg_tune_close_RMSE_pct,n_indices
0,dl,GRU,10,1,98ffd95c2936a288,hid=128 layers=3 drop=0.00 lr=3e-04 bs=16 ep=3...,1.545700,1.882489,4
1,dl,GRU,20,1,4e243d8835e0f454,hid=128 layers=2 drop=0.00 lr=3e-04 bs=16 ep=3...,1.527046,1.903072,4
2,dl,LSTM,10,1,14e15fb52506d645,hid=64 layers=1 drop=0.00 lr=1e-03 bs=16 ep=32...,1.563382,1.911580,4
3,dl,LSTM,20,1,818ed3d710717472,hid=128 layers=2 drop=0.00 lr=3e-04 bs=16 ep=3...,1.564507,1.918009,4
4,dl,MLP,10,1,46a7514e9d3e8d32,hid=128 layers=2 drop=0.00 lr=3e-04 bs=16 ep=3...,1.582951,1.901520,4
5,dl,MLP,20,1,a90257ac87d50781,hid=128 layers=3 drop=0.00 lr=3e-04 bs=16 ep=3...,1.485925,1.862936,4
6,dl,RNN,10,1,2274222ecf3b2f36,hid=128 layers=2 drop=0.00 lr=1e-04 bs=16 ep=3...,1.567103,1.899864,4
7,dl,RNN,20,1,21f8ef9674b41c59,hid=128 layers=1 drop=0.00 lr=3e-04 bs=16 ep=3...,1.470916,1.853008,4
